# Сжатие  и разжатие «Роковых яиц» Булгакова методом BPE

In [43]:
data = open('../../practice/files/Bulgakov_Mihail_Rokovye_yaica.txt', encoding='utf-8').read()
text=data

print(f"Длина текста: {len(text)}")

def generate_ngrams(text, N=3):
     chars = list(text)
     output = []  
     for i in range(len(chars)- N+1):
         output.append(''.join(chars[i:i+N]))
     return output

max_size=1024
table=[]
codes=[chr(code) for code in range(1,500) if chr(code) not in text]

Длина текста: 272247


# Сжатие

In [44]:
for i in range(500):
    d={}
    res=generate_ngrams(text=text,N=2)
    for e in res:
        if not e in d:
            d[e] = 0
        d[e] +=1
    d2 = sorted(d.items(), key=lambda x: x[1], reverse=True)
    best_pair=d2[0][0]
    code=codes[i]

    rows = ''
    for p, c in table + [(best_pair, code)]:
        rows += p + '->' + c + '\n'
    if len(rows.encode('utf-8')) > max_size:
        break

    table.append((best_pair,code))
    text=text.replace(best_pair,code)

print(f"Правил в таблице: {len(table)}")
print(f"Размер таблицы: {len(rows.encode('utf-8'))} байт")
print(f"Сжатая длина: {len(text)}, изначальная: {len(data)}")
print(f"Коэффициент сжатия: {round(len(data) / len(text), 3)}")

Правил в таблице: 130
Размер таблицы: 1025 байт
Сжатая длина: 169757, изначальная: 272247
Коэффициент сжатия: 1.604


# Разжатие

In [45]:
text_back=text
for pair,code in reversed(table):
    text_back=text_back.replace(code,pair)

if text_back==data:
    print("Текст разжался без ошибок")

Текст разжался без ошибок


# Топ-15 правил таблицы

In [46]:
for pair, code in table[:15]:
    print(repr(pair), '->', repr(code))

', ' -> '\x01'
'о ' -> '\x02'
'и ' -> '\x03'
'е ' -> '\x04'
'а ' -> '\x05'
'. ' -> '\x06'
'\n\n' -> '\x07'
'ов' -> '\x08'
'ст' -> '\t'
'ро' -> '\x0b'
'ал' -> '\x0c'
'ер' -> '\r'
' в' -> '\x0e'
'ен' -> '\x0f'
' п' -> '\x10'


# Обсуждение


**• Что в топе таблицы:** Чаще всего встречаются пробелы со знаками препинания и концы слов (', ', 'о ', 'и ', 'е ', '\n\n'), и только потом идут обычные русские слоги ('ов', 'ст', 'ро').

**• Результат сжатия:** За 130 шагов текст ужался с 272 247 до 169 757 символов **(в 1.6 раза)** и разжался обратно без ошибок.